# QUBO + Simulated Annealing (QUBO-SA)

**Objective key:** `qubo_sa` &nbsp;·&nbsp; **Family:** Quantum &nbsp;·&nbsp; **Speed:** Compute-intensive

## Algorithm

Frames portfolio construction as a **combinatorial binary selection** problem:

1. **QUBO formulation** — encode the portfolio objective as an Ising / QUBO energy:  
   `E(x) = -γ·xᵀμ + λ·xᵀΣx + penalty(Σxᵢ ≠ K)`  
   where `x ∈ {0,1}^n` is the binary selection vector and `K` is the cardinality target.
2. **Simulated Annealing** — a classical Metropolis-Hastings temperature schedule searches  
   the binary solution space and selects `K` assets.
3. **Equal weight** on selected assets (or downstream Markowitz if inside hybrid pipeline).

The QUBO structure is identical to that used with **quantum annealers** (D-Wave, IBM Quantum  
annealing); here SA is the solver so no quantum hardware is required.

## Papers

- **Foundational:** Orús, Mugel & Lizaso (2019) — *Quantum Computing for Finance* — Reviews in Physics  
  https://arxiv.org/abs/1811.03975 &nbsp;·&nbsp; **PDF:** https://arxiv.org/pdf/1811.03975
- **Modern (QUBO mapping):** Lucas (2014) — *Ising Formulations of Many NP Problems* — Frontiers in Physics  
  https://arxiv.org/abs/1302.5843 &nbsp;·&nbsp; **PDF:** https://arxiv.org/pdf/1302.5843

## Assumptions

- `mu` and `Sigma` are annualised.
- `K=None` → auto (≈ 60% of assets rounded). Override with an integer to fix cardinality.
- `lambda_risk=1.0`, `gamma=8.0`, `seed=42`.

In [ ]:
import sys
from pathlib import Path

def _find_root() -> Path:
    for p in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
        if (p / "api" / "app.py").is_file():
            return p
    raise RuntimeError("Cannot find repo root")

ROOT = _find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print(f"Repo root: {ROOT}")

In [ ]:
import numpy as np
from services.portfolio_optimizer import run_optimization

rng = np.random.default_rng(42)
n = 12
K = 5  # select 5 out of 12 assets
mu = rng.uniform(0.06, 0.22, n)
cov_raw = rng.standard_normal((n, n))
Sigma = cov_raw @ cov_raw.T / n + np.eye(n) * 0.04
asset_names = [f"ASSET_{i+1:02d}" for i in range(n)]
print(f"{n} assets, K={K} (cardinality target)")

In [ ]:
result = run_optimization(
    returns=mu,
    covariance=Sigma,
    objective="qubo_sa",
    asset_names=asset_names,
    K=K,
    lambda_risk=1.0,
    gamma=8.0,
    seed=42,
)

active = [(name, w) for name, w in zip(asset_names, result.weights) if w > 1e-6]
print(f"\nObjective:       {result.objective}")
print(f"Selected assets: {len(active)} (target K={K})")
print(f"Expected return: {result.expected_return:.4f}")
print(f"Volatility:      {result.volatility:.4f}")
print(f"Sharpe ratio:    {result.sharpe_ratio:.4f}")
print("\nSelected (active) weights:")
for name, w in active:
    print(f"  {name}: {w:.4f}")

In [ ]:
# QUBO-SA with auto K (no cardinality constraint)
result_auto = run_optimization(
    returns=mu, covariance=Sigma, objective="qubo_sa",
    asset_names=asset_names, K=None, lambda_risk=1.0, gamma=8.0, seed=42,
)
active_auto = sum(1 for w in result_auto.weights if w > 1e-6)
print(f"Auto-K result: {active_auto} assets selected, Sharpe={result_auto.sharpe_ratio:.4f}")